# 第二讲：NumPy 数组运算、向量化操作与随机数生成## 为什么量化必须学 NumPy？想象你要计算 5000 只股票 × 10 年的日收益率 —— 那是 **1200 万** 个数据点。用 Python 的 `for` 循环一个个算，等你去喝杯咖啡回来可能还没算完。NumPy 把这 1200 万次运算**一次打包**推给底层 C 代码执行，速度快 **50-200 倍**。> 🧠 **核心思维转变：** 从「遍历每个元素，对它做操作」> 变成「对整列数据做同一件事」。这就是**向量化思维**。**学习目标**- 理解逐元素运算和广播机制——NumPy 最核心的两个「魔法」- 告别 Python 循环，拥抱向量化- 熟练使用 ufunc、布尔索引、`np.where` 条件筛选- 掌握 `np.random.default_rng()` 随机数生成- 完成蒙特卡洛模拟实战> ⚠ **前置知识：** 本讲假定你已学完第一讲（列表、推导式、函数）。> 如果你忘了 `import` 和变量的概念，先回去看第一讲的「第〇步」。---

In [ ]:
# ─── 导入工具箱 ───# numpy 是科学计算的基石，约定俗成缩写为 np# 之后所有 np.xxx 都是在用 NumPy 的功能import numpy as npimport time    # 用来计算代码运行了多久import sys     # 看 Python 版本print(f"NumPy 版本: {np.__version__}")print(f"Python 版本: {sys.version.split()[0]}")

## 2.1 数组运算——NumPy 的核心「魔法」### Python 列表 vs NumPy 数组：本质区别先看一个对比来建立直觉。同样的 `+`，Python 列表和 NumPy 数组做的是完全不同的事：| 操作 | Python 列表 | NumPy 数组 ||------|------------|-----------|| `a + b` | 拼接（把两个列表接在一起） | 逐元素相加 || `a * 3` | 重复（把列表复制 3 遍） | 逐元素乘 3 || `a ** 2` | ❌ 报错！ | 逐元素平方 |> 一句话：**NumPy 数组的运算就像 Excel 里的公式 —— 对整列数据同时生效。**

In [ ]:
# ─── 对比实验：Python 列表 vs NumPy 数组 ───# Python 列表的行为a = [1, 2, 3]b = [4, 5, 6]print("Python list:")print("  a + b =", a + b)       # 拼接！[1,2,3,4,5,6]print("  a * 3 =", a * 3)       # 重复！[1,2,3,1,2,3,1,2,3]# print("  a ** 2 =", a ** 2)   # ❌ 这行会报错！列表不支持幂运算# NumPy 数组的行为a_np = np.array([1, 2, 3])     # np.array() 把 Python 列表转成 NumPy 数组b_np = np.array([4, 5, 6])print("\nNumPy array:")print("  a + b   =", a_np + b_np)     # 逐元素加法：[5, 7, 9]print("  a * 3   =", a_np * 3)         # 逐元素乘 3：[3, 6, 9]print("  a ** 2  =", a_np ** 2)        # 逐元素平方：[1, 4, 9]print("  np.sqrt(a) =", np.sqrt(a_np)) # 逐元素开方print("\n👉 NumPy 更像是「数的运算」，而 Python 列表更像是「容器的操作」。")

### 2.1.1 逐元素运算（Element-wise Operations）**所有算术运算符都是逐元素的。** 这是 NumPy 最基本也最重要的设计原则。

In [ ]:
# ─── 算术运算 + 比较运算 + 标量运算 ───# dtype=np.float64 指定用 64 位浮点数（小数），比默认的整数更精确a = np.array([1, 2, 3, 4, 5], dtype=np.float64)b = np.array([10, 20, 30, 40, 50], dtype=np.float64)print("=== 算术运算 ===（都是逐元素操作）")print("a:", a)print("b:", b)print()print("a + b  =", a + b)       # 每个位置加起来print("a - b  =", a - b)print("a * b  =", a * b)       # ⚠ 这是逐元素乘，不是矩阵乘法！矩阵乘法用 @print("a / b  =", a / b)print("a // b =", a // b)      # 整除（去掉小数部分）print("a % b  =", a % b)       # 取余数print("a ** 2 =", a ** 2)      # 平方print()print("=== 比较运算（返回布尔数组：True/False）===")# 比较运算在量化里极常用：找出 PE<15 的股票、涨幅>5% 的日子等print("a > 2:", a > 2)         # 每个位置判断是否 >2print("a == b:", a == b)       # 每个位置判断是否相等print("a >= b:", a >= b)print()print("=== 与标量的运算（标量自动广播到数组的形状）===")# 一个数字（标量）和一个数组做运算：数字会自动"扩展到"数组的每个元素print("a + 10:", a + 10)       # 每个元素 +10print("a * 2:", a * 2)         # 每个元素 ×2print("1 / a:", 1 / a)         # 1 除以每个元素print("a > 3:", a > 3)         # 每个元素和 3 比较

### 2.1.2 广播机制（Broadcasting）—— NumPy 最强大的特性广播是 NumPy 的「自动对齐」魔法：当两个形状不同的数组运算时，NumPy 自动把小的「拉伸」成大的形状，**不复制数据，不占额外内存**。#### 广播规则（只需记住三条）1. 从尾部维度开始比较（右对齐）2. 两个维度**相等**，或其中一个为 **1**，则兼容3. 不满足 → 报错 `ValueError`#### 图解广播```Case 1: (3, 4) 与标量 10 → 标量被复制 3×4 份      4 列                      4 列  +---+---+---+---+         +---+---+---+---+3 |   |   |   |   |   +   3 | 10| 10| 10| 10|   ← 10 自动填满所有格子  +---+---+---+---+         +---+---+---+---+Case 2: (3, 4) 与 (3, 1) → (3, 1) 的列复制 4 份  (3, 4)      (3, 1)         (3, 4)       广播后  +---+---+---+---+  +---+   +---+---+---+---+  |   |   |   |   |  | 10|   | 10| 10| 10| 10|   ← 每行复制 4 次  |   |   |   |   |+ | 20| = | 20| 20| 20| 20|  |   |   |   |   |  | 30|   | 30| 30| 30| 30|  +---+---+---+---+  +---+   +---+---+---+---+```> ⚠ **常见坑：** 广播只能「拉伸」维度为 1 的方向，不能凭空猜测你想怎么对齐。> 如果两个维度不相等且都不是 1，NumPy 会直接报错，不会尝试「智能匹配」。

In [ ]:
# ─── 广播实战 ───# 例1：矩阵每一行减去该行的均值（数据中心化）# 这是量化里最常用的操作之一：让每行数据的均值归零data = np.array([[1, 2, 3, 4],                 [5, 6, 7, 8],                 [9, 10, 11, 12]], dtype=np.float64)print("原始数据 (3行 × 4列):")print(data)# axis=1 意思是「沿列方向压缩」→ 把每一行的 4 个数字压成 1 个平均值# keepdims=True 意思是「保留被压扁的维度」，结果 shape 是 (3,1) 而非 (3,)# 这样 (3,4) - (3,1) 才能广播！row_means = data.mean(axis=1, keepdims=True)print(f"\n行均值 shape: {row_means.shape}  ← 3行1列，不是 1 行 3 列！")print("行均值:")print(row_means)centered = data - row_means   # (3,4) - (3,1) → 第二维从 1 广播到 4print("\n中心化后（每行均值为 0）:")print(centered)print(f"\n验证每行均值: {centered.mean(axis=1)}  ← 全是 0（或接近 0 的极小值）")print()print("=" * 50)# 例2：对每一列加不同的权重 —— 经典量化场景# 比如：open 权重 0.1, high 权重 0.2, low 权重 0.3, close 权重 0.4weights = np.array([0.1, 0.2, 0.3, 0.4])  # shape (4,) —— 一维数组，4 个元素weighted = data * weights                    # (3,4) * (4,) → 自动广播print("\n加权后（每列乘以不同的系数）:")print(weighted)# 第 0 列全 ×0.1，第 1 列全 ×0.2，以此类推print()print("=" * 50)# 例3：外积 —— 利用广播将两个一维向量变成二维表格# 比如：3 个股票 × 4 个日期 → 生成 3×4 的价格表格x = np.array([1, 2, 3])           # shape (3,)y = np.array([10, 20, 30, 40])    # shape (4,)# np.newaxis 在指定位置插入一个新维度# x[:, np.newaxis] → shape (3, 1) —— 变成列向量# y[np.newaxis, :] → shape (1, 4) —— 变成行向量outer = x[:, np.newaxis] * y[np.newaxis, :]  # (3,1) * (1,4) → (3,4)print("\n外积 (3×4) —— 两个向量的「乘法表」:")print(outer)# 结果：每个位置是 x[i] * y[j]print()print("=" * 50)# 例4：不兼容的广播 → 直接报错print("\n故意制造不兼容的广播:")try:    a = np.ones((3, 4))    # shape (3,4)    b = np.ones((3, 3))    # shape (3,3)    c = a + b              # 4 ≠ 3 且都不为 1 → 报错！except ValueError as e:    print(f"❌ 广播失败: {e}")    print("原因：(3,4) 和 (3,3) 的最后一维 4≠3，无法广播")

### 2.1.3 聚合运算（Aggregation）—— axis 参数的真相许多人被 `axis` 参数困扰。记住这个口诀：> **`axis` 指定的是「被消灭」的维度。**> - `axis=0` → 沿行方向压缩 → 行消失，结果只剩列（每列一个值）> - `axis=1` → 沿列方向压缩 → 列消失，结果只剩行（每行一个值）**量化场景：** 一个二维数组每行是一只股票，每列是一个日期。- `axis=0`：每只股票（行）被压缩 → 得到每个日期的市场平均值- `axis=1`：每个日期（列）被压缩 → 得到每只股票的时间平均值

In [ ]:
# ─── 图解 axis：用 3×3 数组演示 ───arr = np.array([[1, 2, 3],                [4, 5, 6],                [7, 8, 9]])print("原始数组 (3行 × 3列):")print(arr)print()# axis=0：沿第 0 轴（行）方向压缩# 3 行被压成 1 行 → 结果有 3 个值（每列的和）print("axis=0: 沿行方向压缩 →「行消失，列保留」→ 每列求一个值")print("  sum(axis=0):", arr.sum(axis=0),   "← 每列的和 [1+4+7, 2+5+8, 3+6+9]")print("  mean(axis=0):", arr.mean(axis=0), "← 每列的均值")print()# axis=1：沿第 1 轴（列）方向压缩# 3 列被压成 1 列 → 结果有 3 个值（每行的和）print("axis=1: 沿列方向压缩 →「列消失，行保留」→ 每行求一个值")print("  sum(axis=1):", arr.sum(axis=1),   "← 每行的和 [1+2+3, 4+5+6, 7+8+9]")print("  mean(axis=1):", arr.mean(axis=1), "← 每行的均值")print()# ─── keepdims：保留被压缩的维度（方便后续广播）───print("=" * 50)print("=== keepdims=True 的作用 ===\n")# 不加 keepdims：维度被"吃掉"，结果是一维数组without = arr.sum(axis=1)           # shape (3,) —— 只剩一个维度# 加 keepdims：被压缩的维度变成 1，方便后续广播with_kd = arr.sum(axis=1, keepdims=True)  # shape (3, 1) —— 保留二维结构print(f"  sum(axis=1)            → shape: {without.shape}  ← 一维，3 个元素")print(f"  sum(axis=1, keepdims=True) → shape: {with_kd.shape}  ← 二维，3行1列")print()print("有了 keepdims，可以直接用广播做归一化（每行除以行和）:")normalized = arr / arr.sum(axis=1, keepdims=True)# (3,3) / (3,1) → 广播 → 每行除以该行的和print(normalized)print(f"验证每行和: {normalized.sum(axis=1)}  ← 全是 1.0")

In [ ]:
# ─── 常用聚合函数一览 ───arr = np.array([[1, 3, 5], [2, 4, 6]], dtype=np.float64)print("数组:")print(arr, f"  shape={arr.shape}")print()print(f"sum:      {arr.sum():>8}   ← 所有元素的总和")print(f"prod:     {arr.prod():>8}   ← 所有元素的乘积 (1×3×5×2×4×6)")print(f"mean:     {arr.mean():>8.2f} ← 平均值")print(f"std:      {arr.std():>8.2f}  ← 标准差（波动率！量化核心指标）")print(f"var:      {arr.var():>8.2f}  ← 方差（标准差的平方）")print(f"min:      {arr.min():>8}   ← 最小值")print(f"max:      {arr.max():>8}   ← 最大值")print(f"argmin:   {arr.argmin():>8}   ← 最小值在扁平化后的位置")print(f"argmax:   {arr.argmax():>8}   ← 最大值的位置")print(f"cumsum:   {arr.cumsum()}    ← 累积和 [1, 1+3, 1+3+5, ...]")print(f"cumprod:  {arr.cumprod()}   ← 累积积 [1, 1×3, 1×3×5, ...]")print()# ─── 两种写法完全等价 ───a = np.random.randn(3, 4)  # 生成 3×4 的随机正态分布数组print("a.sum() 与 np.sum(a) 完全等价:")print(f"  a.sum()   = {a.sum():.6f}    ← 面向对象写法：数组自己算")print(f"  np.sum(a) = {np.sum(a):.6f}    ← 函数式写法：numpy 帮你算")print("选你顺手的写法就行，没有优劣之分。")

**NumPy 统计函数两种等价写法：**- `a.mean()` ↔ `np.mean(a)`- `a.std()`  ↔ `np.std(a)`- `a.max()`  ↔ `np.max(a)`- `a.min()`  ↔ `np.min(a)`

### 2.1.4 线性代数运算（选学）> ⚠ 如果你刚开始学 NumPy，可以跳过这一节。线性代数在后续的资产组合优化中会用到，但不是入门必备。**关键区分**：- `a * b` → **逐元素乘法**（每个位置单独乘）- `a @ b` 或 `np.dot(a, b)` → **矩阵乘法**（线性代数里的乘法）

In [ ]:
# ─── 矩阵乘法 @ 运算符 ───A = np.array([[1, 2, 3],              [4, 5, 6]])     # shape (2, 3) —— 2行3列B = np.array([[7, 8],              [9, 10],              [11, 12]])      # shape (3, 2) —— 3行2列# (2,3) @ (3,2) → (2,2)。内维 3 必须相等！print("矩阵乘法 A @ B (2×3 @ 3×2 → 2×2):")print(A @ B)print()# 等价写法print("np.dot(A, B) 结果相同:", np.dot(A, B).tolist())print("np.matmul(A, B) 结果相同:", np.matmul(A, B).tolist())print()# 向量点积u = np.array([1, 2, 3])v = np.array([4, 5, 6])# 1×4 + 2×5 + 3×6 = 4+10+18 = 32print(f"向量点积 u·v = {u @ v} = {np.dot(u, v)}")print("几何意义：u 在 v 方向上的投影长度 × v 的长度")

In [ ]:
# ─── 更多线性代数工具（选学）───from numpy import linalg as LAM = np.array([[2, 1],              [1, 2]], dtype=np.float64)print("矩阵 M:")print(M)print()# 求逆矩阵M_inv = LA.inv(M)print("M 的逆矩阵（M @ M_inv = 单位矩阵）:")print(M_inv)print(f"验证 M @ M_inv:\n{M @ M_inv}")print()# 特征值和特征向量eigenvalues, eigenvectors = LA.eig(M)print(f"特征值: {eigenvalues}")print(f"特征向量:\n{eigenvectors}")print()# 行列式print(f"行列式 det(M) = {LA.det(M):.1f}")# SVD 分解U, S, Vt = LA.svd(M)print(f"SVD 奇异值: {S}")

---## 2.2 向量化操作——告别 Python 循环### 2.2.1 什么是向量化？**向量化（Vectorization）** = 用数组级别的操作代替显式的 Python `for` 循环。**为什么快？** 当你在 NumPy 中写 `a + b` 时，循环发生在**编译好的 C 代码层**。Python 解释器每次循环都要做的三件事——字节码解析、类型检查、方法查找——在 C 层全部免掉了。```Python 循环（慢）:           NumPy 向量化（快）:for i in range(len(a)):     c = a + b    c[i] = a[i] + b[i]                                   ↑ C 层一次搞定↑ Python 层循环 n 次  每次都有解释器开销```> 💡 **向量化思维就是你从「怎么操作每个元素」变成「对这列数据做什么」。**

In [ ]:
# ─── 向量化到底快多少？实测对比 ───n = 1_000_000  # 100 万次运算# 方式 A：Python 列表推导式a_list = list(range(n))b_list = list(range(n))t0 = time.perf_counter()c_loop = [a_list[i] + b_list[i] for i in range(n)]t_loop = time.perf_counter() - t0# 方式 B：NumPy 向量化a_np = np.arange(n)     # 和 range() 类似，但返回 NumPy 数组b_np = np.arange(n)t0 = time.perf_counter()c_vec = a_np + b_np     # 一行搞定！t_vec = time.perf_counter() - t0print(f"Python 列表推导式: {t_loop*1000:.1f} ms")print(f"NumPy 向量化:      {t_vec*1000:.1f} ms")print(f"\n⚡ 加速: {t_loop/t_vec:.0f} 倍！")print(f"⚡ 代码量: 2行 vs 1行（且 NumPy 版本更可读）")

### 2.2.2 通用函数（Universal Functions, ufunc）ufunc 是 NumPy 中**逐元素操作**的函数，全部用 C 实现，速度极快。- **一元 ufunc**：接受一个数组，返回一个数组（`np.sqrt`, `np.exp`, `np.sin`）- **二元 ufunc**：接受两个数组，返回一个数组（`np.add`, `np.multiply`, `np.maximum`）**每个算术运算符背后都有一个 ufunc**：`+` → `np.add`，`*` → `np.multiply`……

In [ ]:
# ─── 一元 ufunc（一个输入，一个输出）───x = np.array([0, 1, 2, 3, 10])print("=== 数学函数 ===")print("原数组 x:", x)print("np.sqrt(x): ", np.sqrt(x),   "  ← 开根号（注意：负数开根号得 nan）")print("np.exp(x):  ", np.exp(x),    "  ← e 的 x 次方")print("np.log(x+1):", np.log(x + 1),"  ← 自然对数（+1 避免 log(0) = -inf）")print()print("=== 三角函数 ===")angles = np.array([0, np.pi/2, np.pi])   # 0, 90°, 180°（弧度制）print("角度(弧度):", angles)print("np.sin:", np.sin(angles))          # sin(0)=0, sin(π/2)=1, sin(π)=0print("np.cos:", np.cos(angles))          # cos(0)=1, cos(π/2)=0, cos(π)=-1print()print("=== 取整/符号 ===")vals = np.array([-2.7, -1.2, 0.0, 1.2, 2.7])print("值:     ", vals)print("floor:  ", np.floor(vals),  "  ← 向下取整（往更小的方向）")print("ceil:   ", np.ceil(vals),   "  ← 向上取整（往更大的方向）")print("round:  ", np.round(vals),  "  ← 四舍五入")print("trunc:  ", np.trunc(vals),  "  ← 向零取整（砍掉小数）")print("sign:   ", np.sign(vals),   "  ← 符号：-1/0/+1")

In [ ]:
# ─── 二元 ufunc（两个输入，一个输出）───a = np.array([1, 2, 3, 4])b = np.array([10, 20, 30, 40])print("=== 运算符 = ufunc（功能完全等价）===")print(f"a + b          = {a + b}")print(f"np.add(a, b)   = {np.add(a, b)}    ← + 的背后就是 np.add")print(f"a * b          = {a * b}")print(f"np.multiply(a,b) = {np.multiply(a, b)} ← * 的背后就是 np.multiply")print()print("=== 特别的二元 ufunc ===")print("np.maximum(a, b):", np.maximum(a, b), "  ← 逐位置取较大值")print("np.minimum(a, b):", np.minimum(a, b), "  ← 逐位置取较小值")print("np.power(a, 2):  ", np.power(a, 2),   "  ← a 的 2 次方")print()# ufunc 的 out 参数：结果写入已有数组，避免分配新内存（大数组时省内存）c = np.empty(4)            # 提前分配一块空内存np.add(a, b, out=c)        # 结果直接写入 c，不创建新数组print("np.add(a, b, out=c):", c, "← 结果写入预分配的 c")

### 2.2.3 布尔索引与条件筛选——向量化的 if-else这是 NumPy 中**最常用的数据筛选方式**。在量化分析里无处不在：- 选出 PE < 15 的股票- 找出涨幅 > 5% 的交易日- 标记成交量异常的时段> ⚠ **重要：** 组合条件用 `&` `|` `~`，**不是** `and` `or` `not`！> 因为 `&` 是逐元素逻辑与，`and` 是 Python 关键字（作用于整个 True/False 值）。

In [ ]:
# ─── 布尔索引：三步走 ───data = np.array([15, 8, 42, 23, 7, 91, 56, 4, 33, 18])print("原始数据:", data)print()# 第 1 步：生成布尔掩码（True/False 数组）mask = data > 30print("第 1 步：data > 30 的掩码:")print(" ", mask)# True 的位置就是满足条件的元素# 第 2 步：用掩码筛选filtered = data[mask]print(f"\n第 2 步：筛选结果 data[mask] = {filtered}")print(f"   共 {len(filtered)} 个元素 > 30")# 第 3 步：组合条件（& = 且，| = 或，~ = 非）print()print("=" * 50)print("组合条件：")# 10 < data < 50（两个条件同时满足）mask2 = (data > 10) & (data < 50)#        ↑括号必须加！& 的优先级比 > 高，不加括号会先算 10 & dataprint(f"(data > 10) & (data < 50): {data[mask2]}")# data < 10 或 data > 80mask3 = (data < 10) | (data > 80)print(f"(data < 10) | (data > 80): {data[mask3]}")# 不大于 30（对 mask 取反）mask4 = ~(data > 30)print(f"~(data > 30): {data[mask4]}")print()# ─── 一句话筛选 ───print("一句话筛选（不创建中间变量 mask）:")print(f"data[data > 30]: {data[data > 30]}")

In [ ]:
# ─── np.where()：向量化的 if-else ───# np.where(条件, 条件为真时的值, 条件为假时的值)# 它遍历数组每个位置：条件成立 → 取 A，不成立 → 取 Breturns = np.array([0.05, -0.02, 0.10, -0.08, 0.03, -0.01])# 把负收益标记为 "亏损"，正收益标记为 "盈利"labels = np.where(returns > 0, "盈利📈", "亏损📉")print("收益率:", returns)print("标签:  ", labels)print()# 实战：涨跌幅 > 3% 标记为异常，否则正常changes = np.array([1.2, -5.3, 0.8, 4.7, -2.1, 6.0])flags = np.where(np.abs(changes) > 3, "⚠ 异常", "  正常")for i in range(len(changes)):    print(f"  {changes[i]:+5.1f}%  →  {flags[i]}")

---## 2.3 随机数生成### 为什么量化需要随机数？随机数是量化分析的「素材」：- **蒙特卡洛模拟**：模拟 10000 种股价可能路径- **假设检验**：判断一个策略的收益是不是运气- **合成数据**：在没有真实数据时生成测试数据### 新的 Generator API从 NumPy 1.17 开始，推荐使用 `np.random.default_rng()` 而非旧的 `np.random.seed()`。新 API 的随机数质量更高，且支持可复现的并行随机流。

In [ ]:
# ─── 创建随机数生成器 ───# seed=42 是「随机种子」—— 同样的种子产生同样的随机序列# 这对量化研究至关重要：别人用同样的种子能复现你的结果rng = np.random.default_rng(seed=42)# 如果不用种子，每次运行结果都不同——适合真正需要随机的场景# rng = np.random.default_rng()  # 不设种子print("随机数生成器已创建 (seed=42)")print("每次用同样的 seed，就得到同样的「随机」序列")print("→ 可复现性 = 科学研究的基石\n")# 生成 5 个 0~1 之间的随机数print("rng.random(5):", rng.random(5))print("再运行一次也是同样的 5 个数！")

In [ ]:
# ─── 常用分布一览 ───rng = np.random.default_rng(seed=42)print("=== 常用随机分布 ===\n")# 均匀分布：每个值出现的概率相等print("均匀分布 rng.uniform(0, 100, 5):", rng.uniform(0, 100, 5))print("  → 0~100 之间，每个数概率均等\n")# 正态分布：大部分值集中在均值附近（最常用！）print("正态分布 rng.normal(0, 1, 5):", rng.normal(0, 1, 5))print("  → 均值=0, 标准差=1（标准正态分布）")print("  → 68% 的值落在 -1~1，95% 落在 -2~2\n")# 整数随机print("随机整数 rng.integers(1, 100, 5):", rng.integers(1, 100, 5))print("  → 1~99 之间的随机整数\n")# 从给定列表中随机选stocks = ["茅台", "宁德", "比亚迪", "工行", "招行"]print("随机选择 rng.choice(stocks, 3):", rng.choice(stocks, 3))print("  → 从股票列表中随机抽 3 只")

In [ ]:
# ─── 蒙特卡洛模拟：估计 π ───# 这是理解蒙特卡洛方法最经典的例子# 原理：在正方形里随机撒点，统计落在 1/4 圆内的比例rng = np.random.default_rng(seed=42)n = 100_000  # 撒 10 万个点# 生成 n 个 (x, y) 坐标，范围 [0, 1)x = rng.random(n)y = rng.random(n)# 判断每个点是否在 1/4 圆内：x² + y² ≤ 1inside = (x**2 + y**2) <= 1# inside 是布尔数组，True=1, False=0，求和就是圆内的点数pi_estimate = 4 * inside.sum() / n  # ×4 因为只算了一个象限print(f"估计的 π: {pi_estimate:.6f}")print(f"真实的 π: {np.pi:.6f}")print(f"误差: {abs(pi_estimate - np.pi):.6f}")print(f"\n用 {n:,} 个随机点估计 π，只用了 3 行核心代码！")

## 2.4 小结 & 自检清单| 技能 | ✓ ||------|---|| 理解 Python 列表和 NumPy 数组的本质区别 | ☐ || 逐元素运算：算术、比较、标量广播 | ☐ || 广播机制的三条规则，能解释 (3,4)+(3,1) 为什么可以 | ☐ || axis 参数：axis=0 消灭行，axis=1 消灭列 | ☐ || 向量化的概念 + 知道它为什么快 | ☐ || ufunc：一元/二元，运算符背后的函数 | ☐ || 布尔索引：`&` `|` `~` 组合条件（不是 `and` `or` `not`！） | ☐ || `np.where()` 向量化 if-else | ☐ || `np.random.default_rng(seed=...)` 可复现随机数 | ☐ || 蒙特卡洛基本思路：随机采样 → 统计推断 | ☐ |### 📝 你的笔记区新建一个 Markdown Cell，写下：- 今天学到的最重要的 **3 个 NumPy 概念**- **1 个** 你还不理解的地方（比如广播、axis）- **1 个** 你想用 NumPy 实现的量化小功能---> 🎯 **学完这一讲，你应该能：**> - 用 NumPy 数组代替 Python 列表处理数值数据> - 自信地使用 `axis`、广播、布尔索引> - 理解「向量化」为什么是 NumPy/Pandas 的核心哲学> - 用随机数做简单的蒙特卡洛实验**下一讲预告：** 随机数进阶 + 蒙特卡洛实战（期权定价、股价模拟）。